# AI Document Intelligence — Model Comparison & Benchmark Notebook

**Task**: Automated Document Classification into Invoices, Resumes, and Other.

This notebook benchmarks four classification approaches:
1. **Rule-Based Baseline** (Keyword frequency heuristic)
2. **Multinomial Naive Bayes**
3. **Linear Support Vector Machine (Linear SVM)**
4. **Logistic Regression**

---

In [ ]:
import sys
import os
sys.path.append("..")

from src.utils import load_dataset_from_dir
from src.text_cleaner import TextCleaner
from src.classifier import DocumentClassifier
from src.evaluator import ModelEvaluator
import pandas as pd

print("Environment and modules loaded successfully.")

## 1. Load and Inspect Datasets
We load the partitioned train and test datasets from `../data/`.

In [ ]:
train_texts, train_labels = load_dataset_from_dir("../data/train")
test_texts, test_labels = load_dataset_from_dir("../data/test")

print(f"Training samples: {len(train_texts)}")
print(f"Testing samples:  {len(test_texts)}")

print("Class balance in training set:")
print(pd.Series(train_labels).value_counts())

## 2. Text Preprocessing & Cleaning
Demonstrating systematic text normalization via `TextCleaner`.

In [ ]:
sample_raw = train_texts[0]
clean_result = TextCleaner.clean(sample_raw)

print(f"Original Character Count: {clean_result['char_count_original']}")
print(f"Cleaned Character Count:  {clean_result['char_count_cleaned']}")
print(f"Reduction:                {clean_result['reduction_pct']}%")

## 3. Train Classifiers
Fitting the TF-IDF Vectorizer and training all ML models.

In [ ]:
clf = DocumentClassifier()
clf.train(train_texts, train_labels)
print("Classifier training completed successfully.")

## 4. Evaluation on Independent Test Set
Evaluating Accuracy, Precision, Recall, F1-Score, and Confusion Matrix across all 4 models.

In [ ]:
eval_results = ModelEvaluator.evaluate_all_models(clf, test_texts, test_labels)

df_comp = pd.DataFrame(eval_results["comparison_table"])
print("=== MODEL COMPARISON TABLE ===")
print(df_comp.to_string(index=False))

print(f"\nSelected best model: {eval_results['best_model_name']}")

## 5. Confusion Matrix Diagnostics
Inspecting the confusion matrix for the selected model.

In [ ]:
best_model = eval_results["best_model_name"]
cm = eval_results["detailed_metrics"][best_model]["confusion_matrix"]
df_cm = pd.DataFrame(cm, index=eval_results["classes"], columns=eval_results["classes"])

print(f"Confusion Matrix for {best_model}:")
print(df_cm)

## 6. Key Takeaways and Discussion
- **Rule-Based Baseline vs. ML**: The rule-based baseline performs with high accuracy on standardized documents, while ML models provide probabilistic confidence and handle varied phrasing.
- **Feature Space**: TF-IDF n-grams (1-2) effectively isolate multi-word domain indicators.
- **Missing Field Grace**: Field extractors safely return `Not Found` without raising exceptions when key fields are omitted.